# SupplyForge AI 2.0: KAN-27 - Daily Calendar-SKU Grid Generation



Tài liệu này thực hiện tạo ma trận lưới Daily × SKU hoàn chỉnh từ dữ liệu giao dịch sạch `order_1_cleaned.csv` nhằm biến đổi chuỗi giao dịch ngắt quãng (sparse transactions) thành chuỗi thời gian liên tục (continuous time-series).



Việc phân biệt rõ ràng giữa ngày không phát sinh nhu cầu (genuine zero-demand days) và dữ liệu khuyết thiếu đóng vai trò sống còn trong Feature Engineering (lag features, rolling averages) và huấn luyện mô hình dự báo nhu cầu sau này.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **date_range liên tục**: Xây dựng dải ngày liên tục từ ngày giao dịch nhỏ nhất đến ngày giao dịch lớn nhất sau khi làm sạch.

2. **Active SKUs**: Xác định toàn bộ SKU xuất hiện trong ít nhất một đơn hàng giao dịch thành công (`order_1_cleaned.csv`).

3. **Gộp doanh số hàng ngày**: Tính tổng sản lượng bán hàng `quantity` hàng ngày cho mỗi cặp `(date, itemId)` từ dữ liệu thô.

4. **Cartesian Product**: Sử dụng `pd.MultiIndex.from_product` để tạo đầy đủ tất cả các tổ hợp ngày và Active SKU.

5. **Left Join**: Ghép dữ liệu doanh số hàng ngày vào ma trận lưới rỗng.

6. **Điền khuyết**: Điền các giá trị `quantity` bị thiếu bằng `0` (Zero sales).

7. **Đầu ra**: Tạo file `calendar_daily_sales.csv` chứa các cột `date`, `itemId`, `quantity` và lưu lại.


In [1]:
import pandas as pd

import numpy as np

import json

import os



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'

order_path = os.path.join(DATA_DIR, 'order_1_cleaned.csv')



print("Đang tải dữ liệu đơn hàng sạch...")

df_orders = pd.read_csv(order_path, low_memory=False)

print(f"Tổng số dòng đơn hàng sạch: {len(df_orders)}")


Đang tải dữ liệu đơn hàng sạch...


Tổng số dòng đơn hàng sạch: 39369


In [2]:
print("Đang trích xuất chi tiết sản phẩm (item_list) từ cột JSON data...")

df_orders['time'] = pd.to_datetime(df_orders['time'])

df_orders['date'] = df_orders['time'].dt.date



items_list = []

for idx, row in df_orders.iterrows():

    order_date = row['date']

    data_str = row['data']

    if pd.notna(data_str):

        try:

            data_dict = json.loads(data_str)

            for item in data_dict.get('item_list', []):

                item_id = str(item.get('item_id', ''))

                quantity = int(item.get('model_quantity_purchased', 1))

                if item_id:

                    items_list.append({

                        'date': order_date,

                        'itemId': item_id,

                        'quantity': quantity

                    })

        except Exception as e:

            pass



df_order_items = pd.DataFrame(items_list)

print(f"Tổng số giao dịch chi tiết trích xuất: {len(df_order_items)}")

print(f"Tổng lượng bán hàng trước khi gộp nhóm: {df_order_items['quantity'].sum()}")


Đang trích xuất chi tiết sản phẩm (item_list) từ cột JSON data...


Tổng số giao dịch chi tiết trích xuất: 53020
Tổng lượng bán hàng trước khi gộp nhóm: 65231


In [3]:
print("Đang gộp lượng bán hàng theo Ngày và SKU...")

df_daily_agg = df_order_items.groupby(['date', 'itemId'])['quantity'].sum().reset_index()

print(f"Tổng số dòng sau khi gộp nhóm: {len(df_daily_agg)}")

print(f"Tổng lượng bán hàng sau khi gộp nhóm: {df_daily_agg['quantity'].sum()}")


Đang gộp lượng bán hàng theo Ngày và SKU...
Tổng số dòng sau khi gộp nhóm: 17273
Tổng lượng bán hàng sau khi gộp nhóm: 65231


In [4]:
min_date = df_daily_agg['date'].min()

max_date = df_daily_agg['date'].max()

date_range = pd.date_range(start=min_date, end=max_date, freq='D').date

active_skus = df_daily_agg['itemId'].unique()



print(f"Phạm vi ngày liên tục: Từ {min_date} đến {max_date} ({len(date_range)} ngày)")

print(f"Số lượng Active SKUs (xuất hiện trong ít nhất 1 đơn thành công): {len(active_skus)} SKU")

print(f"Số dòng dự kiến trong ma trận đầy đủ (Days * Active SKUs): {len(date_range) * len(active_skus)}")


Phạm vi ngày liên tục: Từ 2021-04-22 đến 2023-08-31 (862 ngày)
Số lượng Active SKUs (xuất hiện trong ít nhất 1 đơn thành công): 375 SKU
Số dòng dự kiến trong ma trận đầy đủ (Days * Active SKUs): 323250


In [5]:
print("Đang tạo lưới rỗng bằng pd.MultiIndex.from_product...")

grid_index = pd.MultiIndex.from_product([date_range, active_skus], names=['date', 'itemId'])

df_grid = pd.DataFrame(index=grid_index).reset_index()

print(f"Tổng số dòng trong lưới rỗng: {len(df_grid)}")


Đang tạo lưới rỗng bằng pd.MultiIndex.from_product...
Tổng số dòng trong lưới rỗng: 323250


In [6]:
print("Đang ghép nối doanh số thực tế và điền khuyết lượng bán bằng 0...")

df_calendar = pd.merge(df_grid, df_daily_agg, on=['date', 'itemId'], how='left')

df_calendar['quantity'] = df_calendar['quantity'].fillna(0).astype(int)

print(f"Hoàn thành ghép lưới. Tổng số dòng thực tế: {len(df_calendar)}")


Đang ghép nối doanh số thực tế và điền khuyết lượng bán bằng 0...
Hoàn thành ghép lưới. Tổng số dòng thực tế: 323250


In [7]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")



# 1. Kiểm tra tổng dòng

expected_rows = len(date_range) * len(active_skus)

actual_rows = len(df_calendar)

assert actual_rows == expected_rows, f"Lỗi: Số dòng thực tế ({actual_rows}) khác số dòng lý thuyết ({expected_rows})"

print(f"1. Tổng số dòng khớp lý thuyết: Đạt ({actual_rows} dòng)")



# 2. Kiểm tra trùng lặp

duplicates_cnt = df_calendar.duplicated(subset=['date', 'itemId']).sum()

assert duplicates_cnt == 0, f"Lỗi: Phát hiện {duplicates_cnt} bản ghi trùng lặp (date, itemId)"

print("2. Không có trùng lặp (date, itemId) trong lưới: Đạt")



# 3. Kiểm tra giá trị khuyết thiếu

null_qty_cnt = df_calendar['quantity'].isnull().sum()

assert null_qty_cnt == 0, "Lỗi: Phát hiện giá trị Null trong cột quantity"

print("3. Không có giá trị null trong cột quantity: Đạt")



# 4. Kiểm tra tổng lượng bán hàng trước và sau

sum_before = df_daily_agg['quantity'].sum()

sum_after = df_calendar['quantity'].sum()

assert sum_before == sum_after, f"Lỗi: Lượng bán hàng thay đổi trước ({sum_before}) và sau ({sum_after}) ghép lưới"

print(f"4. Tổng lượng bán hàng được bảo toàn: Đạt ({sum_after} sản phẩm)")



# 5. Kiểm tra tính liên tục của ngày

uniq_dates = df_calendar['date'].nunique()

assert uniq_dates == len(date_range), f"Lỗi: Số lượng ngày không khớp ({uniq_dates} vs {len(date_range)})"

print(f"5. Lưới chứa đầy đủ ngày, không bị khuyết: Đạt ({uniq_dates} ngày)")



# 6. Báo cáo tỷ lệ zero-sales

zero_sales_ratio = (df_calendar['quantity'] == 0).mean() * 100

print(f"6. Tỷ lệ Zero-Sales trong lưới: {zero_sales_ratio:.4f}%")



# 7. In ngẫu nhiên 5 bản ghi chèn mới

print("\n7. Mẫu ngẫu nhiên 5 bản ghi zero-sales để kiểm tra bằng mắt:")

print(df_calendar[df_calendar['quantity'] == 0].sample(5, random_state=42))


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

1. Tổng số dòng khớp lý thuyết: Đạt (323250 dòng)
2. Không có trùng lặp (date, itemId) trong lưới: Đạt
3. Không có giá trị null trong cột quantity: Đạt
4. Tổng lượng bán hàng được bảo toàn: Đạt (65231 sản phẩm)
5. Lưới chứa đầy đủ ngày, không bị khuyết: Đạt (862 ngày)
6. Tỷ lệ Zero-Sales trong lưới: 94.6565%

7. Mẫu ngẫu nhiên 5 bản ghi zero-sales để kiểm tra bằng mắt:
              date       itemId  quantity
169971  2022-07-19  10456499522         0
251037  2023-02-20  14483098810         0
60342   2021-09-29  22550232416         0
238863  2023-01-18  16798612591         0
274672  2023-04-24  17467878652         0


In [8]:
print("=== LƯU TRỮ DỮ LIỆU ĐÃ GHÉP LƯỚI ===")

calendar_output_path = os.path.join(DATA_DIR, 'calendar_daily_sales.csv')

df_calendar.to_csv(calendar_output_path, index=False)

print(f"Đã lưu lưới Daily * SKU thành công tại: {calendar_output_path}")


=== LƯU TRỮ DỮ LIỆU ĐÃ GHÉP LƯỚI ===


Đã lưu lưới Daily * SKU thành công tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/calendar_daily_sales.csv
